# GO:BP GPU Hypergeometric ORA Saturation Analysis (study subsampling)

**Environment:** `gpu-kmeans` (Python kernel, cupy/RAPIDS)

Same GPU-vectorized hypergeometric ORA kernel as `gpu_ora_msigdb`, swapping the pathway
database for **GO Biological Process** (`data/pathways/go_bp.Hs.symbols.gmt`, 15,413 terms exported via
`org.Hs.egGO2ALLEGS` + `GO.db`, matching `clusterProfiler::enrichGO(ont="BP")`'s gene-set
semantics including GO-DAG propagation). Same parameters as the MSigDB notebooks: gene
universe = all model genes (`Z.index`), top 1% per LV, min/max gene-set size [10, 50000]
within the universe, one-sided hypergeometric test, BH per LV then minimum across LVs.

1. Discovers the on-disk (rs_pct, K, seed) grid dynamically (full K grid, no capping).
2. Adds a shared rs_pct=100 anchor (K=1728, all 3 seeds) reused from the random-subsampling
   coverage track's full-coverage model (`output/01_model_building/04_archs4/06_bp_coverage_rshall/06_bp_coverage_hall_rs_100`).
3. Saves per-model CSV caches (`rs{rs_pct}_k{k_val}_seed{seed}_bp_gpu_ora.csv` + sibling
   `_meta.csv`) and per-rs_pct-level summary CSVs.

Note: the `n_total_msigdb` field in cache files is a generic "total terms tested" count,
named for parity with the sibling MSigDB notebooks' schema -- here it counts GO:BP terms.


In [1]:
import os
import re
import time

import pandas as pd

REPO_ROOT = "/home/msubirana/Documents/pivlab/clamp-analyses"
os.chdir(REPO_ROOT)

import sys
sys.path.insert(0, os.path.join(REPO_ROOT, "libs"))
import gpu_ora

t_start = time.time()

## Paths

In [2]:
models_dir = "output/01_model_building/04_archs4/08_saturation_study"
output_dir = "output/03_model_biology/00_archs4/08_saturation_study/gpu_ora_bp"

os.makedirs(os.path.join(output_dir, "CLAMPfull"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "CLAMPbase"), exist_ok=True)

## Model grid: discovered dynamically from disk

In [3]:
pattern = re.compile(r"^study_saturation_rs(\d+)_k(\d+)_seed_(\d+)$")

rows = []
for d in sorted(os.listdir(models_dir)):
    m = pattern.match(d)
    if not m:
        continue
    rs_pct, k_val, seed = int(m.group(1)), int(m.group(2)), int(m.group(3))
    z_path_full = os.path.join(models_dir, d, "CLAMPfull_hall", "Z.csv")
    if not os.path.exists(z_path_full):
        continue
    rows.append({
        "rs_pct": rs_pct, "k_val": k_val, "seed": seed, "subdir": d,
        "z_path_full": z_path_full,
        "z_path_base": os.path.join(models_dir, d, "CLAMPbase", "Z.csv"),
    })

model_grid = pd.DataFrame(rows).sort_values(["rs_pct", "k_val", "seed"]).reset_index(drop=True)
print(f"Available models: {len(model_grid)}")
print(model_grid[["rs_pct", "k_val", "seed"]])

Available models: 82
    rs_pct  k_val  seed
0        1     86     1
1        1     86     2
2        1     86     3
3        1    173     1
4        1    173     2
..     ...    ...   ...
77      50    864     1
78      50    864     2
79      50   1296     1
80      50   1296     2
81      50   1728     1

[82 rows x 3 columns]


## Shared rs_pct=100 anchor (K=1728, all 3 seeds)

In [4]:
anchor_models_dir = "output/01_model_building/04_archs4/06_bp_coverage_rshall"
anchor_dir = "06_bp_coverage_hall_rs_100"
anchor_prefix = "hall_coverage"

anchor_rows = []
for seed in [1, 2, 3]:
    seed_dir = os.path.join(anchor_models_dir, anchor_dir, f"{anchor_prefix}_rs100_seed_{seed}")
    z_path_full = os.path.join(seed_dir, "CLAMPfull_hall", "Z.csv")
    if not os.path.exists(z_path_full):
        print(f"SKIP anchor seed{seed} (no Z.csv): {z_path_full}")
        continue
    anchor_rows.append({
        "rs_pct": 100, "k_val": 1728, "seed": seed, "subdir": f"anchor_seed_{seed}",
        "z_path_full": z_path_full,
        "z_path_base": os.path.join(seed_dir, "CLAMPbase", "Z.csv"),
    })

model_grid = pd.concat([model_grid, pd.DataFrame(anchor_rows)], ignore_index=True)
model_grid = model_grid.sort_values(["rs_pct", "k_val", "seed"]).reset_index(drop=True)
print(f"Available models (incl. 100% anchor): {len(model_grid)}")

Available models (incl. 100% anchor): 85


## Load GO:BP gene sets

In [5]:
library = gpu_ora.read_gmt("data/pathways/go_bp.Hs.symbols.gmt")
print(f"GO:BP gene sets loaded: {len(library)}")

GO:BP gene sets loaded: 15413


## Helper: run GPU ORA across the full discovered grid for one model type

In [6]:
def run_model_grid(model_subdir, z_col):
    summary_rows = []

    for _, spec in model_grid.iterrows():
        rs_pct, k_val, seed = int(spec["rs_pct"]), int(spec["k_val"]), int(spec["seed"])
        z_path = spec[z_col]

        if not os.path.exists(z_path):
            print(f"SKIP (no Z.csv): {model_subdir} rs{rs_pct} k{k_val} seed{seed}")
            continue

        cache_path = os.path.join(output_dir, model_subdir, f"rs{rs_pct}_k{k_val}_seed{seed}_bp_gpu_ora.csv")
        meta_path = os.path.join(output_dir, model_subdir, f"rs{rs_pct}_k{k_val}_seed{seed}_meta.csv")

        if os.path.exists(cache_path) and os.path.exists(meta_path):
            print(f"Loading cached: {model_subdir} rs{rs_pct} k{k_val} seed{seed}")
            meta = pd.read_csv(meta_path).iloc[0]
        else:
            print(f"Running GPU ORA: {model_subdir} rs{rs_pct} k{k_val} seed{seed}")
            t0 = time.time()
            res = gpu_ora.run_gpu_ora_for_model(z_path, library, min_size=10, max_size=50000, pct=0.01)
            print(f"  done in {time.time()-t0:.2f}s")
            res["terms_padj"].rename_axis("term").reset_index(name="padj_min_across_lvs").to_csv(cache_path, index=False)
            meta = pd.Series({k: v for k, v in res.items() if k != "terms_padj"})
            meta.to_frame().T.to_csv(meta_path, index=False)

        summary_rows.append({
            "model_type": model_subdir, "rs_pct": rs_pct, "k_val": k_val, "seed": seed,
            "n_samples": meta["n_samples"], "n_lvs": meta["n_lvs"],
            "n_top_genes": meta["n_top_genes"], "n_total_msigdb": meta["n_total_msigdb"],
        })

    summary_df = pd.DataFrame(summary_rows).sort_values(["rs_pct", "k_val", "seed"]).reset_index(drop=True)

    for rs_pct in sorted(summary_df["rs_pct"].unique()):
        pct_df = summary_df[summary_df["rs_pct"] == rs_pct]
        pct_csv_path = os.path.join(output_dir, model_subdir, f"results_pct{rs_pct}_bp_gpu_ora.csv")
        pct_df.to_csv(pct_csv_path, index=False)
        print(f"Saved {model_subdir} rs{rs_pct}: {len(pct_df)} models -> {pct_csv_path}")

    return summary_df

## Run GPU ORA: CLAMPfull

In [7]:
results_full_df = run_model_grid("CLAMPfull", "z_path_full")
print(f"Collected {len(results_full_df)} CLAMPfull rows")
print(results_full_df)

Loading cached: CLAMPfull rs1 k86 seed1
Loading cached: CLAMPfull rs1 k86 seed2
Loading cached: CLAMPfull rs1 k86 seed3
Loading cached: CLAMPfull rs1 k173 seed1
Loading cached: CLAMPfull rs1 k173 seed2
Loading cached: CLAMPfull rs1 k173 seed3
Loading cached: CLAMPfull rs1 k432 seed1
Loading cached: CLAMPfull rs1 k432 seed2
Loading cached: CLAMPfull rs1 k432 seed3
Loading cached: CLAMPfull rs1 k864 seed1
Loading cached: CLAMPfull rs1 k864 seed2
Loading cached: CLAMPfull rs1 k864 seed3
Loading cached: CLAMPfull rs1 k1296 seed1
Loading cached: CLAMPfull rs1 k1296 seed2
Loading cached: CLAMPfull rs1 k1296 seed3
Loading cached: CLAMPfull rs1 k1728 seed2
Loading cached: CLAMPfull rs1 k1728 seed3
Loading cached: CLAMPfull rs5 k86 seed1
Loading cached: CLAMPfull rs5 k86 seed2
Loading cached: CLAMPfull rs5 k86 seed3
Loading cached: CLAMPfull rs5 k173 seed1
Loading cached: CLAMPfull rs5 k173 seed2
Loading cached: CLAMPfull rs5 k173 seed3
Loading cached: CLAMPfull rs5 k432 seed1
Loading cached: C

  done in 3.77s
Running GPU ORA: CLAMPfull rs50 k86 seed2


  done in 0.85s
Running GPU ORA: CLAMPfull rs50 k173 seed1


  done in 1.01s
Running GPU ORA: CLAMPfull rs50 k173 seed2


  done in 0.96s
Running GPU ORA: CLAMPfull rs50 k432 seed1


  done in 1.18s
Running GPU ORA: CLAMPfull rs50 k432 seed2


  done in 1.19s
Running GPU ORA: CLAMPfull rs50 k864 seed1


  done in 1.49s
Running GPU ORA: CLAMPfull rs50 k864 seed2


  done in 1.50s
Loading cached: CLAMPfull rs50 k1296 seed1
Loading cached: CLAMPfull rs50 k1296 seed2
Loading cached: CLAMPfull rs50 k1728 seed1
Loading cached: CLAMPfull rs100 k1728 seed1
Loading cached: CLAMPfull rs100 k1728 seed2
Loading cached: CLAMPfull rs100 k1728 seed3
Saved CLAMPfull rs1: 17 models -> output/03_model_biology/00_archs4/08_saturation_study/gpu_ora_bp/CLAMPfull/results_pct1_bp_gpu_ora.csv
Saved CLAMPfull rs5: 18 models -> output/03_model_biology/00_archs4/08_saturation_study/gpu_ora_bp/CLAMPfull/results_pct5_bp_gpu_ora.csv
Saved CLAMPfull rs10: 18 models -> output/03_model_biology/00_archs4/08_saturation_study/gpu_ora_bp/CLAMPfull/results_pct10_bp_gpu_ora.csv
Saved CLAMPfull rs25: 18 models -> output/03_model_biology/00_archs4/08_saturation_study/gpu_ora_bp/CLAMPfull/results_pct25_bp_gpu_ora.csv
Saved CLAMPfull rs50: 11 models -> output/03_model_biology/00_archs4/08_saturation_study/gpu_ora_bp/CLAMPfull/results_pct50_bp_gpu_ora.csv
Saved CLAMPfull rs100: 3 models 

## Run GPU ORA: CLAMPbase

In [8]:
results_base_df = run_model_grid("CLAMPbase", "z_path_base")
print(f"Collected {len(results_base_df)} CLAMPbase rows")
print(results_base_df)

Loading cached: CLAMPbase rs1 k86 seed1
Loading cached: CLAMPbase rs1 k86 seed2
Loading cached: CLAMPbase rs1 k86 seed3
Loading cached: CLAMPbase rs1 k173 seed1
Loading cached: CLAMPbase rs1 k173 seed2
Loading cached: CLAMPbase rs1 k173 seed3
Loading cached: CLAMPbase rs1 k432 seed1
Loading cached: CLAMPbase rs1 k432 seed2
Loading cached: CLAMPbase rs1 k432 seed3
Loading cached: CLAMPbase rs1 k864 seed1
Loading cached: CLAMPbase rs1 k864 seed2
Loading cached: CLAMPbase rs1 k864 seed3
Loading cached: CLAMPbase rs1 k1296 seed1
Loading cached: CLAMPbase rs1 k1296 seed2
Loading cached: CLAMPbase rs1 k1296 seed3
Loading cached: CLAMPbase rs1 k1728 seed2
Loading cached: CLAMPbase rs1 k1728 seed3
Loading cached: CLAMPbase rs5 k86 seed1
Loading cached: CLAMPbase rs5 k86 seed2
Loading cached: CLAMPbase rs5 k86 seed3
Loading cached: CLAMPbase rs5 k173 seed1
Loading cached: CLAMPbase rs5 k173 seed2
Loading cached: CLAMPbase rs5 k173 seed3
Loading cached: CLAMPbase rs5 k432 seed1
Loading cached: C

  done in 0.85s
Running GPU ORA: CLAMPbase rs50 k86 seed2


  done in 0.85s
Running GPU ORA: CLAMPbase rs50 k173 seed1


  done in 1.00s
Running GPU ORA: CLAMPbase rs50 k173 seed2


  done in 0.94s
Running GPU ORA: CLAMPbase rs50 k432 seed1


  done in 1.15s
Running GPU ORA: CLAMPbase rs50 k432 seed2


  done in 1.21s
Running GPU ORA: CLAMPbase rs50 k864 seed1


  done in 1.49s
Running GPU ORA: CLAMPbase rs50 k864 seed2


  done in 1.51s
Loading cached: CLAMPbase rs50 k1296 seed1
Loading cached: CLAMPbase rs50 k1296 seed2
Loading cached: CLAMPbase rs50 k1728 seed1
Loading cached: CLAMPbase rs100 k1728 seed1
Loading cached: CLAMPbase rs100 k1728 seed2
Loading cached: CLAMPbase rs100 k1728 seed3
Saved CLAMPbase rs1: 17 models -> output/03_model_biology/00_archs4/08_saturation_study/gpu_ora_bp/CLAMPbase/results_pct1_bp_gpu_ora.csv
Saved CLAMPbase rs5: 18 models -> output/03_model_biology/00_archs4/08_saturation_study/gpu_ora_bp/CLAMPbase/results_pct5_bp_gpu_ora.csv
Saved CLAMPbase rs10: 18 models -> output/03_model_biology/00_archs4/08_saturation_study/gpu_ora_bp/CLAMPbase/results_pct10_bp_gpu_ora.csv
Saved CLAMPbase rs25: 18 models -> output/03_model_biology/00_archs4/08_saturation_study/gpu_ora_bp/CLAMPbase/results_pct25_bp_gpu_ora.csv
Saved CLAMPbase rs50: 11 models -> output/03_model_biology/00_archs4/08_saturation_study/gpu_ora_bp/CLAMPbase/results_pct50_bp_gpu_ora.csv
Saved CLAMPbase rs100: 3 models 

## Run GPU ORA: GTEx and Recount2 (full-sample, single models)

Not part of the ARCHS4 rs/K subsampling grid -- one CLAMPfull_hall and one CLAMPbase
model per dataset, using all samples. Reused as external reference points alongside the
ARCHS4 saturation curve.

In [9]:
external_models = {
    "GTEx": {
        "CLAMPfull": "output/01_model_building/02_gtex/10_CLAMP_hall/CLAMPfull_hall/Z.csv",
        "CLAMPbase": "output/01_model_building/02_gtex/01_CLAMP/CLAMPbase/Z.csv",
    },
    "Recount2": {
        "CLAMPfull": "output/01_model_building/03_recount2/01_recount2_hall/CLAMPfull_hall/Z.csv",
        "CLAMPbase": "output/01_model_building/03_recount2/00_recount2/CLAMPbase/Z.csv",
    },
}

external_rows = []
for dataset, model_paths in external_models.items():
    dataset_dir = os.path.join(output_dir, dataset)
    os.makedirs(dataset_dir, exist_ok=True)

    for model_type, z_path in model_paths.items():
        if not os.path.exists(z_path):
            print(f"SKIP (no Z.csv): {dataset} {model_type} -> {z_path}")
            continue

        cache_path = os.path.join(dataset_dir, f"{model_type}_bp_gpu_ora.csv")
        meta_path = os.path.join(dataset_dir, f"{model_type}_meta.csv")

        if os.path.exists(cache_path) and os.path.exists(meta_path):
            print(f"Loading cached: {dataset} {model_type}")
            meta = pd.read_csv(meta_path).iloc[0]
        else:
            print(f"Running GPU ORA: {dataset} {model_type}")
            t0 = time.time()
            res = gpu_ora.run_gpu_ora_for_model(z_path, library, min_size=10, max_size=50000, pct=0.01)
            print(f"  done in {time.time()-t0:.2f}s")
            res["terms_padj"].rename_axis("term").reset_index(name="padj_min_across_lvs").to_csv(cache_path, index=False)
            meta = pd.Series({k: v for k, v in res.items() if k != "terms_padj"})
            meta.to_frame().T.to_csv(meta_path, index=False)

        external_rows.append({
            "dataset": dataset, "model_type": model_type,
            "n_samples": meta["n_samples"], "n_lvs": meta["n_lvs"],
            "n_top_genes": meta["n_top_genes"], "n_total_msigdb": meta["n_total_msigdb"],
        })

results_external_df = pd.DataFrame(external_rows)
results_external_df.to_csv(os.path.join(output_dir, "results_external_bp_gpu_ora.csv"), index=False)
print(f"Collected {len(results_external_df)} external rows")
print(results_external_df)

Running GPU ORA: GTEx CLAMPfull


  done in 1.98s
Running GPU ORA: GTEx CLAMPbase


  done in 1.51s
Running GPU ORA: Recount2 CLAMPfull


  done in 0.43s
Running GPU ORA: Recount2 CLAMPbase


  done in 0.38s
Collected 4 external rows
    dataset model_type  n_samples  n_lvs  n_top_genes  n_total_msigdb
0      GTEx  CLAMPfull      17382    578          217            6377
1      GTEx  CLAMPbase      17382    578          217            6377
2  Recount2  CLAMPfull      37032    724           60            4802
3  Recount2  CLAMPbase      37032    724           60            4802


In [10]:
print(f"Total notebook time: {(time.time()-t_start)/60:.1f} min")

Total notebook time: 0.4 min
